# Chapter 5 figure generator

This notebook generates the eight proposed Chapter 5 figures directly from
versioned experiment data in this repository. It does not embed experimental
measurements in the plotting code. Every numerical value is loaded from the
source CSV or JSON files listed below and checked before plotting.

Outputs are written to `Draft/figures/` as 300-dpi PNG files and vector SVG
files. Run all cells from anywhere inside the repository.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch


def find_repository_root(start=None):
    """Find the repository root without relying on a machine-specific path."""
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        marker = candidate / "reports" / "week4-week7-master-comparison.csv"
        if marker.is_file():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run the notebook from inside unlearning-thesis."
    )


REPO_ROOT = find_repository_root()
OUTPUT_DIR = REPO_ROOT / "Draft" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "identities": REPO_ROOT / "Week 2/data/synthetic_facts_v1/identities.csv",
    "facts": REPO_ROOT / "Week 2/data/synthetic_facts_v1/facts_long.csv",
    "evaluation": REPO_ROOT / "Week 2/data/synthetic_facts_v1/qa_eval_all.csv",
    "dataset_metadata": REPO_ROOT / "Week 2/data/synthetic_facts_v1/metadata.json",
    "strict_baseline": REPO_ROOT / "Week 3.5/results/qwen05_high_accuracy_baseline/metrics.json",
    "direct_ascent_samples": REPO_ROOT / "Week 4/results/gradient_ascent_unlearning_v1/results/all_before_after_results.csv",
    "direct_ascent_history": REPO_ROOT / "Week 4/results/gradient_ascent_unlearning_v1/results/unlearning_history.csv",
    "preservation_sweep": REPO_ROOT / "Week 5/results/retain_regularized_unlearning_resumable_v1/results/candidate_best_summary.csv",
    "cross_experiment": REPO_ROOT / "reports/week4-week7-master-comparison.csv",
}

missing = [str(path.relative_to(REPO_ROOT)) for path in SOURCES.values() if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required source files:\n" + "\n".join(missing))

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "legend.fontsize": 9.5,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
    }
)

PALETTE = {
    "navy": "#17365D",
    "blue": "#2F75B5",
    "teal": "#2A9D8F",
    "green": "#70AD47",
    "gold": "#E9C46A",
    "orange": "#F4A261",
    "red": "#C94C4C",
    "gray": "#7F8C8D",
    "light": "#EEF3F8",
}


def save_figure(fig, stem):
    """Save a print-quality raster and an editable vector copy."""
    png_path = OUTPUT_DIR / f"{stem}.png"
    svg_path = OUTPUT_DIR / f"{stem}.svg"
    fig.savefig(png_path, dpi=300, bbox_inches="tight", pad_inches=0.12)
    fig.savefig(svg_path, format="svg", bbox_inches="tight", pad_inches=0.12)
    plt.close(fig)
    return png_path, svg_path


def percent_label(value):
    return f"{value:.1f}%" if abs(value - round(value)) > 1e-8 else f"{value:.0f}%"


print(f"Repository root: {REPO_ROOT}")
print(f"Figure output:   {OUTPUT_DIR}")

Repository root: C:\Users\hanna\Documents\Unlearning Thesis\unlearning-thesis
Figure output:   C:\Users\hanna\Documents\Unlearning Thesis\unlearning-thesis\Draft\figures


## Load and validate the documented experiment data

The assertions are deliberate: if a source file changes incompatibly, the
notebook stops instead of silently drawing a misleading figure.

In [2]:
identities = pd.read_csv(SOURCES["identities"])
facts = pd.read_csv(SOURCES["facts"])
evaluation = pd.read_csv(SOURCES["evaluation"])
with SOURCES["dataset_metadata"].open(encoding="utf-8") as handle:
    dataset_metadata = json.load(handle)
with SOURCES["strict_baseline"].open(encoding="utf-8") as handle:
    baseline_metrics = json.load(handle)
direct_samples = pd.read_csv(SOURCES["direct_ascent_samples"])
direct_history = pd.read_csv(SOURCES["direct_ascent_history"])
preservation_sweep = pd.read_csv(SOURCES["preservation_sweep"])
cross_experiment = pd.read_csv(SOURCES["cross_experiment"])

summary = dataset_metadata["summary"]
assert len(identities) == summary["num_identities"] == 100
assert len(facts) == summary["num_facts"] == 500
assert len(evaluation) == summary["num_eval_examples"] == 1500
assert identities["split"].value_counts().to_dict() == {"retain": 80, "forget": 20}
assert set(direct_samples["model_stage"]) == {"before_unlearning", "after_gradient_ascent"}
assert set(direct_history["epoch"]) == set(range(1, 9))
assert len(preservation_sweep) == 9
assert cross_experiment["forget_heldout"].notna().all()
assert cross_experiment["retain_heldout"].notna().all()
assert cross_experiment["general"].notna().all()

source_inventory = pd.DataFrame(
    {
        "data role": SOURCES.keys(),
        "repository-relative source": [str(path.relative_to(REPO_ROOT)) for path in SOURCES.values()],
    }
)
source_inventory

,data role,repository-relative source
0,identities,Week 2\data\synthetic_facts_v1\identities.csv
1,facts,Week 2\data\synthetic_facts_v1\facts_long.csv
2,evaluation,Week 2\data\synthetic_facts_v1\qa_eval_all.csv
3,dataset_metadata,Week 2\data\synthetic_facts_v1\metadata.json
4,strict_baseline,Week 3.5\results\qwen05_high_accuracy_baseline...
5,direct_ascent_samples,Week 4\results\gradient_ascent_unlearning_v1\r...
6,direct_ascent_history,Week 4\results\gradient_ascent_unlearning_v1\r...
7,preservation_sweep,Week 5\results\retain_regularized_unlearning_r...
8,cross_experiment,reports\week4-week7-master-comparison.csv


## Figure 1 - End-to-end experimental pipeline

In [3]:
forget_n = int((identities["split"] == "forget").sum())
retain_n = int((identities["split"] == "retain").sum())
lora_forget = 100 * baseline_metrics["lora_after_training"]["forget_all"]["contains_value"]
lora_retain = 100 * baseline_metrics["lora_after_training"]["retain_all"]["contains_value"]

pipeline = [
    ("Synthetic data", f"{len(identities)} identities\n{len(facts)} facts"),
    ("Target partition", f"{forget_n} forget\n{retain_n} retain"),
    ("Strict baseline learning", f"Forget {lora_forget:.1f}%\nRetain {lora_retain:.1f}%"),
    ("Candidate unlearning", "Ascent + preservation\nLoRA updates"),
    ("Conflict control", "Projection, adaptive\nweights, rollback"),
    ("Guarded selection", "Forgetting, retain,\nand general checks"),
    ("Full evaluation", f"{len(evaluation)} synthetic\n+ 50 general prompts"),
]

fig, ax = plt.subplots(figsize=(16, 5.6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
box_w, box_h, y0 = 0.125, 0.34, 0.36
x_positions = np.linspace(0.015, 0.86, len(pipeline))
fills = ["#DDEBF7", "#E2F0D9", "#FFF2CC", "#FCE4D6", "#E4DFEC", "#D9EAD3", "#DDEBF7"]

for index, ((title, detail), x0, fill) in enumerate(zip(pipeline, x_positions, fills), start=1):
    box = FancyBboxPatch(
        (x0, y0), box_w, box_h,
        boxstyle="round,pad=0.012,rounding_size=0.018",
        facecolor=fill, edgecolor=PALETTE["navy"], linewidth=1.5,
    )
    ax.add_patch(box)
    ax.text(x0 + box_w / 2, y0 + 0.235, title, ha="center", va="center", weight="bold", fontsize=10.5)
    ax.text(x0 + box_w / 2, y0 + 0.105, detail, ha="center", va="center", fontsize=9.5, linespacing=1.35)
    ax.text(x0 + 0.012, y0 + box_h - 0.035, str(index), ha="center", va="center", color="white", fontsize=8.5,
            bbox=dict(boxstyle="circle,pad=0.22", facecolor=PALETTE["navy"], edgecolor="none"))
    if index < len(pipeline):
        arrow = FancyArrowPatch(
            (x0 + box_w + 0.004, y0 + box_h / 2),
            (x_positions[index] - 0.004, y0 + box_h / 2),
            arrowstyle="-|>", mutation_scale=15, linewidth=1.4, color=PALETTE["gray"],
        )
        ax.add_patch(arrow)

ax.text(0.5, 0.88, "End-to-End Machine-Unlearning Experimental Pipeline", ha="center", va="center", fontsize=16, weight="bold")
ax.text(0.5, 0.19, "All displayed counts and baseline accuracies are loaded from the repository data.",
        ha="center", va="center", fontsize=10, color="#555555")
save_figure(fig, "01_experimental_pipeline")

(WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/01_experimental_pipeline.png'),
 WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/01_experimental_pipeline.svg'))

## Figure 2 - Forget/retain composition at three dataset levels

In [4]:
partition_counts = pd.DataFrame(
    {
        "Forget": [
            (identities["split"] == "forget").sum(),
            (facts["split"] == "forget").sum(),
            (evaluation["split"] == "forget").sum(),
        ],
        "Retain": [
            (identities["split"] == "retain").sum(),
            (facts["split"] == "retain").sum(),
            (evaluation["split"] == "retain").sum(),
        ],
    },
    index=["Identities", "Facts", "Evaluation prompts"],
)
partition_pct = partition_counts.div(partition_counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10.5, 4.9))
y = np.arange(len(partition_counts))
left = np.zeros(len(partition_counts))
for name, color in [("Forget", PALETTE["red"]), ("Retain", PALETTE["teal"])]:
    values = partition_pct[name].to_numpy()
    bars = ax.barh(y, values, left=left, color=color, height=0.56, label=name)
    for row_index, (bar, pct, count) in enumerate(zip(bars, values, partition_counts[name])):
        ax.text(left[row_index] + pct / 2, bar.get_y() + bar.get_height() / 2,
                f"{int(count):,}\n({pct:.0f}%)", ha="center", va="center", color="white", weight="bold")
    left += values

ax.set_yticks(y, partition_counts.index)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Share of each dataset level (%)")
ax.set_title("Forget and Retain Partitions Across the Synthetic Dataset")
ax.legend(ncol=2, frameon=False, loc="lower center", bbox_to_anchor=(0.5, -0.3))
ax.xaxis.grid(True, color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
save_figure(fig, "02_dataset_partitions")

(WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/02_dataset_partitions.png'),
 WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/02_dataset_partitions.svg'))

## Figure 3 - Strict baseline learning, before and after LoRA training

In [5]:
metric_specs = [
    ("Forget\nall", "forget_all"),
    ("Retain\nall", "retain_all"),
    ("Forget\nheld-out", "forget_heldout_paraphrases"),
    ("Retain\nheld-out", "retain_heldout_paraphrases"),
    ("Forget\nseen", "forget_seen_prompts"),
    ("Retain\nseen", "retain_seen_prompts"),
]
labels = [item[0] for item in metric_specs] + ["General\ncontrol"]
before = [100 * baseline_metrics["base_before_training"][key]["contains_value"] for _, key in metric_specs]
after = [100 * baseline_metrics["lora_after_training"][key]["contains_value"] for _, key in metric_specs]
before.append(baseline_metrics["general_control"]["base_before_training_contains_value_percentage"])
after.append(baseline_metrics["general_control"]["lora_after_training_contains_value_percentage"])

fig, ax = plt.subplots(figsize=(13.2, 6.2))
x = np.arange(len(labels))
width = 0.36
bars_before = ax.bar(x - width / 2, before, width, color="#A9B7C6", label="Base model before training")
bars_after = ax.bar(x + width / 2, after, width, color=PALETTE["blue"], label="LoRA model after training")
for bars in (bars_before, bars_after):
    for bar in bars:
        value = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, value + 1.4, percent_label(value), ha="center", va="bottom", fontsize=8.8)

ax.set_xticks(x, labels)
ax.set_ylim(0, 112)
ax.set_ylabel("Contains-value accuracy (%)")
ax.set_title("Strict High-Accuracy Baseline: Before and After LoRA Training")
ax.legend(frameon=False, ncol=2, loc="upper center")
ax.yaxis.grid(True, color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
save_figure(fig, "03_strict_baseline_learning")

(WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/03_strict_baseline_learning.png'),
 WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/03_strict_baseline_learning.svg'))

## Figure 4 - Multi-objective unlearning architecture

In [6]:
fig, ax = plt.subplots(figsize=(14, 7.6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis("off")


def node(x, y, w, h, title, subtitle="", face="#FFFFFF", edge=PALETTE["navy"]):
    patch = FancyBboxPatch(
        (x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor=face, edgecolor=edge, linewidth=1.5,
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h * 0.64, title, ha="center", va="center", weight="bold", fontsize=10.5)
    if subtitle:
        ax.text(x + w / 2, y + h * 0.29, subtitle, ha="center", va="center", fontsize=9.3, linespacing=1.25)
    return patch


def connect(x1, y1, x2, y2, color=PALETTE["gray"], style="-|>"):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle=style, mutation_scale=15,
                                 linewidth=1.35, color=color, connectionstyle="arc3,rad=0"))


node(0.4, 5.45, 2.1, 1.05, "Forget examples", "Target facts", face="#FCE4D6", edge=PALETTE["red"])
node(0.4, 2.85, 2.1, 1.05, "Retain examples", "Preserved facts", face="#E2F0D9", edge=PALETTE["teal"])
node(0.4, 0.5, 2.1, 1.05, "Frozen reference", "Strict learned model", face="#EDEDED", edge=PALETTE["gray"])

node(3.45, 5.25, 2.65, 1.45, "Forgetting objective", r"$L_f=-\mathrm{CE}_f$" + "\nDirect-ascent gradient", face="#FCE4D6", edge=PALETTE["red"])
node(3.45, 2.55, 2.65, 1.45, "Retention objective", r"$L_r=\mathrm{CE}_r+\lambda_{KL}D_{KL}$" + "\nPreservation gradient", face="#E2F0D9", edge=PALETTE["teal"])

node(7.05, 3.65, 2.75, 1.65, "Gradient controller", "PCGrad projection\nAdaptive weighting", face="#E4DFEC", edge="#8064A2")
node(10.65, 3.65, 2.75, 1.65, "Guarded update", "Threshold checks\nRollback when violated", face="#FFF2CC", edge="#BF9000")
node(10.65, 1.15, 2.75, 1.25, "LoRA adapter", "Accepted model state", face="#DDEBF7", edge=PALETTE["blue"])

connect(2.5, 5.98, 3.45, 5.98, PALETTE["red"])
connect(2.5, 3.38, 3.45, 3.38, PALETTE["teal"])
connect(2.5, 1.02, 3.45, 2.85, PALETTE["gray"])
connect(6.1, 5.98, 7.05, 4.88, PALETTE["red"])
connect(6.1, 3.28, 7.05, 4.08, PALETTE["teal"])
connect(9.8, 4.48, 10.65, 4.48, "#8064A2")
connect(12.02, 3.65, 12.02, 2.4, "#BF9000")
connect(10.65, 1.78, 9.3, 3.65, PALETTE["blue"])

ax.text(0.4, 7.35, "Repository-Documented Multi-Objective Unlearning Architecture", fontsize=16, weight="bold")
ax.text(8.55, 6.55, "Reconciles conflicting\nforget/retain gradients",
        ha="center", va="center", fontsize=9.5, color="#555555")
save_figure(fig, "04_unlearning_objective_architecture")

(WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/04_unlearning_objective_architecture.png'),
 WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/04_unlearning_objective_architecture.svg'))

## Figure 5 - Cross-experiment forgetting-utility Pareto view

In [7]:
pareto = cross_experiment.copy()
pareto["experiment_name"] = pareto["method"].str.replace("_", " ", regex=False).str.strip().str.title()
pareto = pareto.sort_values(["phase", "experiment_name"]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13.5, 8.4))
xmin = max(0, pareto["forget_heldout"].min() - 8)
xmax = min(100, pareto["forget_heldout"].max() + 5)
ymin = max(0, pareto["retain_heldout"].min() - 7)
ymax = 100
ax.axvspan(xmin, 45, ymin=(82 - ymin) / (ymax - ymin), ymax=1, color="#D9EAD3", alpha=0.55,
           label="Target region")
ax.axvline(45, color=PALETTE["red"], linestyle="--", linewidth=1.3)
ax.axhline(82, color=PALETTE["teal"], linestyle="--", linewidth=1.3)

normalizer = mcolors.Normalize(vmin=pareto["general"].min(), vmax=pareto["general"].max())
scatter = ax.scatter(
    pareto["forget_heldout"], pareto["retain_heldout"],
    c=pareto["general"], cmap="viridis", norm=normalizer,
    s=150, edgecolor="white", linewidth=1.2, zorder=3,
)
label_offsets = {
    1: (-8, -13), 2: (0, 0), 3: (0, 0), 4: (0, 0), 5: (0, 0),
    6: (0, 0), 7: (0, 0), 8: (9, 5), 9: (-13, 5),
}
for index, row in pareto.iterrows():
    point_id = index + 1
    offset = label_offsets[point_id]
    annotation_style = dict(
        xytext=offset, textcoords="offset points", ha="center", va="center",
        color="white", fontsize=8.5, weight="bold",
    )
    if point_id in {1, 8, 9}:
        annotation_style["bbox"] = dict(
            boxstyle="circle,pad=0.2", facecolor="#333333", edgecolor="white", linewidth=0.6
        )
    ax.annotate(str(point_id), (row["forget_heldout"], row["retain_heldout"]), **annotation_style)

legend_lines = [Line2D([], [], linestyle="none", label=f"{i + 1}. {name}") for i, name in enumerate(pareto["experiment_name"])]
ax.legend(handles=legend_lines, ncol=2, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14),
          handlelength=0, handletextpad=0, columnspacing=1.5)
cbar = fig.colorbar(scatter, ax=ax, pad=0.015)
cbar.set_label("General-control accuracy (%)")
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_xlabel("Forget held-out accuracy (%) - lower is better")
ax.set_ylabel("Retain held-out accuracy (%) - higher is better")
ax.set_title("Forgetting-Utility Trade-off Across Full Experimental Evaluations")
ax.text(44.2, 82.6, "Forget <= 45%\nRetain >= 82%", ha="right", va="bottom", fontsize=9, color="#3D6B3D")
ax.grid(True, color="#EAEAEA", linewidth=0.8)
ax.set_axisbelow(True)
save_figure(fig, "05_forgetting_utility_pareto")

(WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/05_forgetting_utility_pareto.png'),
 WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/05_forgetting_utility_pareto.svg'))

## Figure 6 - Hyperparameter sensitivity of retain-regularized unlearning

In [8]:
sweep = preservation_sweep.copy()
sweep["candidate_short"] = sweep["candidate_id"].str.extract(r"^(c\d+)", expand=False)
sweep["candidate_label"] = (
    sweep["candidate_id"].str.replace(r"^c\d+_", "", regex=True).str.replace("_", " ", regex=False)
)
lr_norm = mcolors.LogNorm(vmin=sweep["learning_rate"].min(), vmax=sweep["learning_rate"].max())
marker_by_retain = {1.0: "o", 2.0: "s", 4.0: "D"}

fig, ax = plt.subplots(figsize=(12.8, 7.7))
for retain_weight, group in sweep.groupby("retain_weight"):
    ax.scatter(
        group["forget_heldout_selection_percentage"],
        group["retain_heldout_selection_percentage"],
        c=group["learning_rate"], cmap="plasma", norm=lr_norm,
        s=150 + 260 * group["kl_weight"], marker=marker_by_retain[float(retain_weight)],
        edgecolor="black", linewidth=0.7, alpha=0.9, zorder=3,
    )
candidate_offsets = {
    "c01": (4, -14), "c02": (5, -13), "c03": (4, -14),
    "c04": (5, 8), "c05": (5, 8), "c06": (5, -13),
    "c07": (5, 8), "c08": (5, -16), "c09": (-18, 8),
}
for _, row in sweep.iterrows():
    ax.annotate(
        row["candidate_short"],
        (row["forget_heldout_selection_percentage"], row["retain_heldout_selection_percentage"]),
        xytext=candidate_offsets[row["candidate_short"]], textcoords="offset points", fontsize=8.2,
        weight="bold", ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.16", facecolor="white", edgecolor="#777777", alpha=0.82),
    )

scalar_map = plt.cm.ScalarMappable(norm=lr_norm, cmap="plasma")
scalar_map.set_array([])
cbar = fig.colorbar(scalar_map, ax=ax, pad=0.015)
cbar.set_label("Learning rate")
cbar.set_ticks(sorted(sweep["learning_rate"].unique()))
cbar.set_ticklabels([f"{value:.0e}" for value in sorted(sweep["learning_rate"].unique())])

retain_handles = [
    Line2D([0], [0], marker=marker_by_retain[value], color="none", markerfacecolor="#BFBFBF",
           markeredgecolor="black", markersize=9, label=f"Retain weight = {value:g}")
    for value in sorted(marker_by_retain)
]
kl_handles = [
    plt.scatter([], [], s=150 + 260 * value, facecolor="none", edgecolor="#555555", label=f"KL weight = {value:g}")
    for value in sorted(sweep["kl_weight"].unique())
]
legend_a = ax.legend(handles=retain_handles, frameon=False, loc="lower right", title="Marker shape")
ax.add_artist(legend_a)
ax.legend(handles=kl_handles, frameon=False, loc="upper left", title="Marker area")
ax.axhline(85, color=PALETTE["teal"], linestyle="--", linewidth=1.2)
ax.text(sweep["forget_heldout_selection_percentage"].min(), 85.18, "Retain eligibility threshold (85%)",
        ha="left", va="bottom", fontsize=9, color=PALETTE["teal"])
candidate_handles = [
    Line2D([], [], linestyle="none", label=f"{row.candidate_short}: {row.candidate_label}")
    for row in sweep.sort_values("candidate_short").itertuples()
]
fig.legend(handles=candidate_handles, ncol=3, frameon=False, loc="lower center",
           bbox_to_anchor=(0.5, -0.12), handlelength=0, handletextpad=0, columnspacing=1.5, fontsize=8.5)
ax.set_xlabel("Forget held-out selection accuracy (%) - lower is better")
ax.set_ylabel("Retain held-out selection accuracy (%) - higher is better")
ax.set_title("Retain-Regularized Unlearning: Full Candidate Sensitivity Sweep")
ax.grid(True, color="#EAEAEA", linewidth=0.8)
ax.set_axisbelow(True)
save_figure(fig, "06_hyperparameter_sensitivity")

(WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/06_hyperparameter_sensitivity.png'),
 WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/06_hyperparameter_sensitivity.svg'))

## Figure 7 - Direct-gradient-ascent trajectory

In [9]:
history = direct_history.sort_values("epoch").copy()
eligible = history["retain_eligible"].astype(str).str.lower().eq("true")
eligible_rows = history.loc[eligible]
selected_index = eligible_rows["selection_score"].idxmax()
selected = history.loc[selected_index]

fig, ax = plt.subplots(figsize=(11.5, 6.4))
ax.plot(history["epoch"], history["forget_train_percentage"], marker="o", linewidth=2.2,
        color=PALETTE["red"], label="Forget training accuracy")
ax.plot(history["epoch"], history["retain_train_sample_percentage"], marker="s", linewidth=2.2,
        color=PALETTE["teal"], label="Retain sample accuracy")
ax.axhline(85, color="#555555", linestyle="--", linewidth=1.2, label="Retain threshold (85%)")
ax.axvspan(history.loc[eligible, "epoch"].min() - 0.35, history.loc[eligible, "epoch"].max() + 0.35,
           color="#D9EAD3", alpha=0.45, label="Eligible epochs")
ax.scatter([selected["epoch"]], [selected["forget_train_percentage"]], s=220, facecolor="none",
           edgecolor=PALETTE["navy"], linewidth=2.0, zorder=5)
ax.annotate(
    f"Selected epoch {int(selected['epoch'])}\nForget {selected['forget_train_percentage']:.0f}%, retain {selected['retain_train_sample_percentage']:.0f}%",
    (selected["epoch"], selected["forget_train_percentage"]), xytext=(24, -24), textcoords="offset points",
    arrowprops=dict(arrowstyle="->", color=PALETTE["navy"]), fontsize=9.5,
)
ax.set_xticks(history["epoch"])
ax.set_ylim(0, 105)
ax.set_xlabel("Training epoch")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Direct Gradient Ascent: Forgetting and Retention Trajectory")
ax.legend(frameon=False, ncol=2, loc="lower center")
ax.grid(True, color="#EAEAEA", linewidth=0.8)
ax.set_axisbelow(True)
save_figure(fig, "07_direct_gradient_ascent_trajectory")

(WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/07_direct_gradient_ascent_trajectory.png'),
 WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/07_direct_gradient_ascent_trajectory.svg'))

## Figure 8 - Category-level effects of direct gradient ascent

In [10]:
category = (
    direct_samples.loc[direct_samples["eval_split"].isin(["forget", "retain"])]
    .groupby(["eval_split", "category", "model_stage"], as_index=False)["contains_value"]
    .mean()
)
category["accuracy"] = 100 * category["contains_value"]
category_order = ["access_phrase", "favorite_city", "lab_number", "research_topic", "secret_code"]
category_labels = {
    "access_phrase": "Access phrase",
    "favorite_city": "Favorite city",
    "lab_number": "Lab number",
    "research_topic": "Research topic",
    "secret_code": "Secret code",
}

fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.4), sharey=True)
panel_specs = [
    ("forget", "Forget identities - lower after is better", PALETTE["red"]),
    ("retain", "Retain identities - higher after is better", PALETTE["teal"]),
]
y = np.arange(len(category_order))

for ax, (split_name, title, after_color) in zip(axes, panel_specs):
    subset = category.loc[category["eval_split"] == split_name]
    pivot = subset.pivot(index="category", columns="model_stage", values="accuracy").reindex(category_order)
    before_values = pivot["before_unlearning"].to_numpy()
    after_values = pivot["after_gradient_ascent"].to_numpy()
    for yi, before_value, after_value in zip(y, before_values, after_values):
        ax.plot([before_value, after_value], [yi, yi], color="#B8B8B8", linewidth=3, zorder=1)
    ax.scatter(before_values, y, s=90, color=PALETTE["navy"], label="Before unlearning", zorder=3)
    ax.scatter(after_values, y, s=90, color=after_color, label="After gradient ascent", zorder=3)
    for yi, before_value, after_value in zip(y, before_values, after_values):
        ax.text(before_value, yi - 0.22, percent_label(before_value), ha="center", va="bottom", fontsize=8.5,
                color=PALETTE["navy"])
        ax.text(after_value, yi + 0.22, percent_label(after_value), ha="center", va="top", fontsize=8.5,
                color=after_color)
    ax.set_xlim(0, 105)
    ax.set_xlabel("Contains-value accuracy (%)")
    ax.set_title(title, fontsize=12)
    ax.xaxis.grid(True, color="#EAEAEA", linewidth=0.8)
    ax.set_axisbelow(True)

axes[0].set_yticks(y, [category_labels[item] for item in category_order])
axes[0].invert_yaxis()
category_handles = [
    Line2D([0], [0], marker="o", linestyle="none", color=PALETTE["navy"], markersize=9, label="Before unlearning"),
    Line2D([0], [0], marker="o", linestyle="none", color=PALETTE["red"], markersize=9, label="After: forget identities"),
    Line2D([0], [0], marker="o", linestyle="none", color=PALETTE["teal"], markersize=9, label="After: retain identities"),
]
fig.legend(handles=category_handles, frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Fact-Category Effects of Direct Gradient Ascent", fontsize=15, weight="bold", y=1.01)
fig.tight_layout(rect=(0, 0.07, 1, 0.98))
save_figure(fig, "08_fact_category_effects")

(WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/08_fact_category_effects.png'),
 WindowsPath('C:/Users/hanna/Documents/Unlearning Thesis/unlearning-thesis/Draft/figures/08_fact_category_effects.svg'))

## Generated files

In [11]:
generated = sorted(path.relative_to(REPO_ROOT) for path in OUTPUT_DIR.glob("*.*"))
assert len(generated) == 16, f"Expected 16 output files, found {len(generated)}"
pd.DataFrame({"generated file": [str(path) for path in generated]})

,generated file
0,Draft\figures\01_experimental_pipeline.png
1,Draft\figures\01_experimental_pipeline.svg
2,Draft\figures\02_dataset_partitions.png
3,Draft\figures\02_dataset_partitions.svg
4,Draft\figures\03_strict_baseline_learning.png
5,Draft\figures\03_strict_baseline_learning.svg
6,Draft\figures\04_unlearning_objective_architec...
7,Draft\figures\04_unlearning_objective_architec...
8,Draft\figures\05_forgetting_utility_pareto.png
9,Draft\figures\05_forgetting_utility_pareto.svg
